In [15]:
from arch import arch_model
import pandas as pd
import numpy as np
from cvxpy import length
import scipy.stats as stats

# Backtesting

Backtesting refers to the process of evaluating how a model or strategy would have performed on historical data, **using only the information that would have been available at each point in time**.

The goal is to assess how reliable a model is likely to be on new, unseen data (out-of-sample), rather than on the data it was fit on.

The general procedure works as follows:

1. Fit the model using only data available up to time `t` (or use previously estimated parameters as of time `t`).
2. Generate a forecast for `t+1` (or further ahead, depending on the horizon).
3. Observe what actually happened at `t+1`.
4. Compare the forecast to the realized outcome.
5. Repeat this process across many time points to build a performance series or summary statistic.

In the context of VaR/CVaR modeling, backtesting specifically asks: *are the model's risk forecasts consistent with realized outcomes?* For example, if VaR is estimated at a 99% confidence level, roughly 1% of realized returns should exceed that threshold over time. Backtesting checks whether this holds in practice, which is exactly what tests like **Kupiec** (checks whether the total number of exceedances matches the expected rate) and **Christoffersen** (checks whether exceedances are independent over time, rather than clustering together) are designed to evaluate.

Two common approaches are used to structure this process:

- **Rolling window**: a fixed-size window (e.g. the last 500 days) that shifts forward at each step, dropping the oldest observation and adding the newest one, then refitting the model.
- **Expanding window**: the window grows over time, always using all historical data from the start up to the current point, then refitting the model.

####  How to impelement expanding and rolling window in our framework was handled in 07_arch_garch.ipynb

The choice between them depends on how quickly the underlying market regime changes and the specific goals of the analysis; for highly persistent models like GARCH, rolling windows are often preferred, since they limit the influence of outdated regimes on current estimates.

## Look-Ahead Bias, In-Sample Fit, and Out-of-Sample Fit

A critical requirement for valid backtesting is avoiding **look-ahead bias**, meaning the model must never have access to information from the future relative to the point being forecasted.

This becomes relevant when considering a specific scenario: taking a model that has already been fit on four years of data, then truncating the dataset to one year earlier (without retraining) and forecasting the value immediately following that truncated point.

This approach is **not logically valid** as a backtesting method, because the model's parameters (μ, ω, α, β) were estimated using the *entire* four-year sample, including data from after the truncation point.

In effect, the model already "knows" about volatility events, shocks, or calm periods that occurred after the forecasted date, since those events influenced the maximum likelihood estimation of the parameters. A real investor or risk manager standing at that earlier date would not have had access to this future information.

This kind of experiment isn't entirely meaningless, however; it can still be useful as a check of **in-sample fit quality**, essentially asking: "given the best possible parameters estimated from all available data, how good is even a one-step-ahead forecast?" But this measures something fundamentally different from **out-of-sample forecast performance**, which is what backtesting is actually meant to evaluate. Because of this distinction, proper VaR/CVaR backtesting requires a rolling or expanding window approach, refitting the model using only data available up to each point in time, rather than relying on a single model fit on the full sample. Using a full-sample model instead would invalidate any conclusions drawn from Kupiec or Christoffersen tests, since the underlying forecasts would be contaminated by look-ahead bias.

In [16]:
tickers = ["AAPL", "JPM", "XOM", "KO"]

close_price = pd.read_csv("../data_general/4_ticker_data.csv")

close_price["Date"] = pd.to_datetime(close_price["Date"])
close_price = close_price.set_index("Date").sort_index()

percent_return = close_price.pct_change()

percent_return_scaled = percent_return * 100  # optimizer warned us that values in decimal form 0.002 might result in gradient saturation

# Question


## Rolling Window VAR Empirical Forecast





In [17]:

percent_return_scaled = percent_return * 100

period_of_forecast = 252
window_size = 250
weekday_indexes = percent_return_scaled.sort_index(ascending=False).index


forecasts = {}

for target_date in weekday_indexes[:period_of_forecast]:

    #only difference is here:
    start_date = target_date - pd.Timedelta(days=window_size)

    returns_with_cutoff_date = percent_return_scaled.loc[start_date:target_date]

    basic_gm = arch_model(
        returns_with_cutoff_date["AAPL"].dropna(),
        p=1,
        q=1,
        mean="constant",
        vol="garch",
        dist="normal"
    )

    res = basic_gm.fit(disp="off")

    gm_forecast = res.forecast(horizon=1)
    #forecast_value = gm_forecast.variance.loc[target_date, "h.1"] #it was variance expectation

    std_resid = res.std_resid
    quantile_residue_5percent = std_resid.quantile(0.05)

    # transforming this quantile value to return terms
    mean_forecast = gm_forecast.mean
    variance_forecast = gm_forecast.variance

    VaR_empirical = mean_forecast.values + np.sqrt(variance_forecast).values * quantile_residue_5percent


    forecast_index = target_date + pd.Timedelta(days=1)
    forecasts[forecast_index] = VaR_empirical

variance_forecasts_rolling = pd.DataFrame.from_dict(
    forecasts,
    orient="index",
    columns=["VarForecast"]
).sort_index()



ValueError: Must pass 2-d input. shape=(252, 1, 1)

####  Realized Losses as Benchmark


In [ ]:
percent_loss = -percent_return
percent_loss.head(1)

https://www.mathworks.com/help/risk/overview-of-var-backtesting.html


* For many portfolios, especially trading portfolios, VaR is computed daily. At the closing of the following day, the actual profits and losses for the portfolio are known and can be compared to the VaR estimated the day before.

* You can use this daily data to assess the performance of VaR models, which is the goal of VaR backtesting



#### Let $x$ be number of failures (VaR was higher then expectation)
#### And N days / number of observations:


# Binomial Test

* It only cares about the exceedance indicator: did the loss breach VaR or not (0 or 1), not by how much

* Under a correctly calibrated VaR model, each day is like a Bernoulli trial with success probability $p = 1 - VaR   level$ (e.g., p = 0.01 for 99% VaR)

*  Over N days you expect N * p breaches. The test statistic:

The test statistics:

$$Z_{bin} = \frac{x - Np}{\sqrt{Np(1-p)}} $$

tells you whether the observed breach count x is a plausible draw from Binomial(N, p), using the normal approximation for large N.

### Remembering how to read Z-test:

** Magnitude tells you significance**

Since $Z_{bin}$ approximately follows a standard normal distribution, it's compared against critical values:

$$
\text{Reject if } |Z_{bin}| > z_{1-\alpha/2}
$$

Common thresholds:

| Test confidence level | Critical value $z_{1-\alpha/2}$ |
|---|---|
| 90% | 1.645 |
| 95% | 1.96 |
| 99% | 2.576 |

Example: if you're testing at the 95% confidence level and $|Z_{bin}| > 1.96$, **you reject the model (the exception count is statistically significantly different from what's expected).**

## Rejecting H0
means observed breach rate $=/$ expected breach rate (We have more breach)





In [ ]:
def loss_exceeding_days(
    realized_loss: pd.DataFrame,
    var_value_df: pd.DataFrame,
    start_date: str | pd.Timestamp ,
    end_date: str | pd.Timestamp
) -> dict:

    if not realized_loss.index.equals(var_value_df.index):
        missing_in_var = realized_loss.index.difference(var_value_df.index)
        missing_in_loss = var_value_df.index.difference(realized_loss.index)
        raise ValueError(
            f"Index mismatch. Dates missing in var_value_df: {list(missing_in_var)}, "
            f"dates missing in realized_loss: {list(missing_in_loss)}"
        )

    result = {}

    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    #getting amount of weekdays
    number_of_days_in_df= len(realized_loss.loc[start_date:end_date])

    #calculating days of breach
    realized_loss = realized_loss.loc[start_date:end_date]
    var_value_df = var_value_df.loc[start_date:end_date]

    days_breach_status = realized_loss["loss"] > var_value_df["varForecast"]

    #calculating time difference between breaches

    indexes_of_days_exceeded = np.where(days_breach_status)[0]

    #this dataset doesnt have the time till the first breach
    days_between_exceedances = np.diff(indexes_of_days_exceeded)

    first_breach = np.argmax(days_breach_status.values) + 1

    days_between_exceedances = np.insert(days_between_exceedances, 0, first_breach)

    result["days_exceeded_boolean"] = days_breach_status
    result["X_days_exceeded_number"] = days_breach_status.sum()
    result["N_window_size_by_weekdays_in_df"] = number_of_days_in_df
    result["days_between_exceedances"] = days_between_exceedances



    return result




In [ ]:
def binomial_test_single_window(
    start_date: str | pd.Timestamp,
    end_date: str | pd.Timestamp,
    realized_loss: pd.DataFrame,
    var_value_df: pd.DataFrame,
    var_confidence: float,
    test_confidence: float = 0.95
) -> bool:


    preprocess= loss_exceeding_days(realized_loss,var_value_df,start_date,end_date)
    X_days_exceeded_number = preprocess["X_days_exceeded_number"]
    N_window_size = preprocess["N_window_size_by_weekdays_in_df"]

    p = 1 - var_confidence
    # if var confidence is %90,possibility of loss being less than var 0.90


    expected_breach = N_window_size * p #over n days n*p breaches expected

    test_statistic = (X_days_exceeded_number - (expected_breach))/np.sqrt(expected_breach * (1-p))

    test_statistic = np.abs(test_statistic)

    alpha = 1 - test_confidence

    two_tail_test_possibility = 1 - alpha/2
    threshold = stats.norm.ppf(two_tail_test_possibility)

    if(test_statistic > threshold):
        return False #false: BAD, H0 rejected, model is not
    else:
        return True



# Traffic Light Test

* A variation on the binomial test proposed by the Basel Committee is the traffic light test or three zones test.

**Step 1: Compute the cumulative binomial probability**

$$
F(x) = P(X \leq x) = \sum_{k=0}^{x} \binom{N}{k} p^k (1-p)^{N-k}
$$

where $N$ = total observations, $p$ = 1 - VaR level (e.g., 0.01 for 99% VaR). **This is the probability of seeing *x or fewer* exceptions if the model is actually correct.**

**Step 2: Classify x into a zone based on F(x)**

$$
\begin{cases}
\text{Green} & \text{if } F(x) < 0.95 \\
\text{Yellow} & \text{if } 0.95 \leq F(x) < 0.9999 \\
\text{Red} & \text{if } F(x) \geq 0.9999
\end{cases}
$$

**Concrete example**

99% VaR, N = 250 days, so p = 0.01, expected exceptions Np = 2.5.

| x (exceptions) | F(x) | Zone |
|---|---|---|
| 0-4 | < 0.95 | Green |
| 5 | ≈ 0.958 | Yellow |
| 6 | ≈ 0.992 | Yellow |
| 9 | ≈ 0.9997 | Yellow |
| 10 | ≈ 0.99997 | Red


In [ ]:
def basel_traffic_light_test_single_window(
    start_date: str | pd.Timestamp,
    end_date: str | pd.Timestamp,
    realized_loss: pd.DataFrame,
    var_value_df: pd.DataFrame,
    var_confidence: float,
) -> int:

    preprocess= loss_exceeding_days(realized_loss,var_value_df,start_date,end_date)
    X_days_exceeded_number = preprocess["X_days_exceeded_number"]
    N_window_size = preprocess["N_window_size_by_weekdays_in_df"]

    p = 1 - var_confidence
    # if var confidence is %90,possibility of loss being less than var 0.90


    cumulative_binomial_probability = stats.binom.cdf(X_days_exceeded_number, N_window_size, p)

    threshold_green = 0.95
    threshold_red = 0.9999

    if cumulative_binomial_probability < threshold_green:
        return 1
    elif threshold_green <= cumulative_binomial_probability <threshold_red :
        return 2
    else:
        return 3



# Kupiec’s POF and TUFF Tests

## 1. POF (Proportion of Failures) test


$$
LR_{POF} = -2\log\left[\frac{(1-p)^{N-x}p^{x}}{\left(1-\frac{x}{N}\right)^{N-x}\left(\frac{x}{N}\right)^{x}}\right] \sim \chi^2_{(1)}
$$


- **Numerator (restricted/null model):** assumes the true exception probability is exactly $p$ (what the VaR level implies, e.g. 0.01 for 99% VaR)
- **Denominator (unrestricted/alternative model):** assumes the true exception probability is whatever was actually observed, $\hat{p} = x/N$

If the model is correctly calibrated, these two likelihoods should be close, so the ratio ≈ 1, and $-2\log(\text{ratio})$ ≈ 0. If $\hat p$ (observed) is very different from $p$ (theoretical), the numerator likelihood is much smaller than the denominator's best-fit likelihood, the ratio shrinks, and $-2\log(\cdot)$ grows large.

Both POF and $Z_{bin}$ test the same null hypothesis ($H_0: p_{true} = p$). In fact, asymptotically, $Z_{bin}^2 \approx LR_{POF}$, since a squared standard normal is a chi-square with 1 df. So Kupiec's POF is the *likelihood ratio version* of the same binomial coverage test, just expressed through log-likelihoods instead of standardized differences.

**Decision rule**

$$
\text{Reject model if } LR_{POF} > \chi^2_{1,\, 1-\alpha}
$$

Common critical values: 3.84 (α = 5%), 6.63 (α = 1%).



In [ ]:
def kupiec_pof_single_window(
    start_date: str | pd.Timestamp,
    end_date: str | pd.Timestamp,
    realized_loss: pd.DataFrame,
    var_value_df: pd.DataFrame,
    var_confidence: float,
    test_confidence: float = 0.95,
    detailed_output: bool = False
) -> bool | dict:

    preprocess = loss_exceeding_days(realized_loss, var_value_df, start_date, end_date)
    X_days_exceeded_number = preprocess["X_days_exceeded_number"]
    N_window_size = preprocess["N_window_size_by_weekdays_in_df"]

    p = 1 - var_confidence

    p_hat = X_days_exceeded_number / N_window_size

    log_numerator = (N_window_size - X_days_exceeded_number) * np.log(1 - p) + X_days_exceeded_number * np.log(p)

    if X_days_exceeded_number == 0:
        log_denominator = N_window_size * np.log(1 - p_hat)
    elif X_days_exceeded_number == N_window_size:
        log_denominator = X_days_exceeded_number * np.log(p_hat)
    else:
        log_denominator = (N_window_size - X_days_exceeded_number) * np.log(1 - p_hat) + X_days_exceeded_number * np.log(p_hat)

    test_statistic = -2 * (log_numerator - log_denominator)

    alpha = 1 - test_confidence
    threshold = stats.chi2.ppf(1 - alpha, df=1)

    reject_h0 = test_statistic > threshold

    if detailed_output:
        output = {}
        output["test_statistic"] = test_statistic
        output["threshold"] = threshold
        output["p_hat"] = p_hat
        output["X_days_exceeded_number"] = X_days_exceeded_number
        output["N_window_size"] = N_window_size
        output["reject_h0"] = reject_h0
        return output

    return not reject_h0


## 2. TUFF (Time Until First Failure) test

$$
LR_{TUFF} = -2\log\left[\frac{p(1-p)^{n-1}}{\left(\frac{1}{n}\right)\left(1-\frac{1}{n}\right)^{n-1}}\right] \sim \chi^2_{(1)}
$$

**What's different here**

Instead of counting *how many* exceptions happened over N days, TUFF only asks: how many days, $n$, did it take until the *first* exception occurred? This follows a **geometric distribution**: probability that the first failure happens exactly on day $n$ is $p(1-p)^{n-1}$.

- **Numerator:** likelihood under the theoretical $p$ (implied by VaR level)
- **Denominator:** likelihood under the empirical estimate $\hat{p} = 1/n$ (i.e., "if failures happen at rate 1 per n days, what's the best-fit probability?")

## **Intuition:**

* if the first exception comes way too early (small $n$) relative to what $p$ would predict, that's a red flag that the model underestimates risk.
* If it never comes for a very long time, that's a sign the model may be too conservative.

**Why TUFF is weaker than POF**

* TUFF discards everything after the first failure.
* If your first exception happens on day 5 and everything looks fine after that (or falls apart with 20 more exceptions), TUFF only "sees" day 5.
* This is a big information loss, especially over a long backtest window (like a 250-day setup), most of the signal about model quality lives in the pattern of *repeated* failures, not just the first one.

**TBFI test as the fix**

TBFI (time between failures) extends TUFF to look at *all* the gaps between consecutive failures, not just the first one, using the same geometric/likelihood-ratio machinery repeated across every inter-failure interval. It recovers the information TUFF throws away.


In [ ]:
def kupiec_tuff_single_window(
    start_date: str | pd.Timestamp,
    end_date: str | pd.Timestamp,
    realized_loss: pd.DataFrame,
    var_value_df: pd.DataFrame,
    var_confidence: float,
    test_confidence: float = 0.95
) -> bool:

    preprocess = loss_exceeding_days(realized_loss, var_value_df, start_date, end_date)
    X_days_exceeded_number = preprocess["X_days_exceeded_number"]

    if X_days_exceeded_number == 0:
        raise ValueError("Kupiec TUFF test is undefined when there is no exceedance")

    n = preprocess["days_between_exceedances"][0] #days till first breach

    p = 1 - var_confidence

    log_numerator = np.log(p) + (n - 1) * np.log(1 - p)

    if n == 1:
        log_denominator = np.log(1/n)
    else:
        log_denominator = np.log(1/n) + (n - 1) * np.log(1 - 1/n)

    test_statistic = -2 * (log_numerator - log_denominator)

    alpha = 1 - test_confidence
    threshold = stats.chi2.ppf(1 - alpha, df=1)

    return test_statistic <= threshold

## **3. Haas's TBF (Time Between Failures) / Mixed Kupiec Test**

Haas (2001) fixes this by applying TUFF's geometric distribution logic to **each** consecutive pair of exceptions separately, then summing them all up.

$$
LR_{TBFI} = -2\sum_{i=1}^{x}\log\left[\frac{p(1-p)^{n_i-1}}{\left(\frac{1}{n_i}\right)\left(1-\frac{1}{n_i}\right)^{n_i-1}}\right]
$$

Here $n_i$ = number of days between the $(i-1)$-th and $i$-th exception (for the first exception, the time from the start until that exception).

**Intuitively:** think of each exception as its own "TUFF test" relative to the exception before it. If there are 5 exceptions, you compute 5 separate TUFF-like likelihood ratios (each with its own $n_i$), then sum them. This way you're using the entire exception history, not just the first one.

**Why the chi-square degrees of freedom = x (number of exceptions):**

Each $n_i$ term contributes its own $\chi^2_{(1)}$ (following TUFF logic), and since these are treated as independent, their degrees of freedom add up when summed: $x$ independent $\chi^2_{(1)}$ terms sum to $\chi^2_{(x)}$.


## Decision Rule

$$
LR_{TBF} = LR_{POF} + LR_{TBFI} \sim \chi^2_{(x+1)}
$$


$$
\text{Reject model if } LR_{TBF} > \chi^2_{x+1,\, 1-\alpha}
$$

### $LR_{TBF}$ and  $LR_{TBFI}$ difference?

* $LR_{POF}$ is Proportion of Failures test, described in first section
* $LR_{TBFI}$ is firmula is given here
* Decision rule, $LR_{TBF}$ is calculated via difference of them.


### Decomposing the result: which part is failing?**

Since $LR_{TBF} = LR_{POF} + LR_{TBFI}$, when you get a rejection, break it back down into its two components to diagnose *why*:

- **High $LR_{POF}$, low $LR_{TBFI}$** → the problem is frequency (too many or too few total exceptions), timing pattern itself looks fine
- **Low $LR_{POF}$, high $LR_{TBFI}$** → total count is right, but the exceptions are clustered/irregularly spaced (timing problem), classic sign of volatility clustering not captured by the model
- **Both high** → model fails on both counts, likely a fundamentally misspecified model

This decomposition is the main practical value of TBF over plain POF, it tells you not just "reject or not" but roughly *where* the model is breaking down.


In [ ]:
def haas_tbf_single_window(
    start_date: str | pd.Timestamp,
    end_date: str | pd.Timestamp,
    realized_loss: pd.DataFrame,
    var_value_df: pd.DataFrame,
    var_confidence: float,
    test_confidence: float = 0.95
) -> bool:

    preprocess = loss_exceeding_days(realized_loss, var_value_df, start_date, end_date)
    X_days_exceeded_number = preprocess["X_days_exceeded_number"]



    n_t = preprocess["days_between_exceedances"]  # N IS NOT SCALAR ANYMORE!

    p = 1 - var_confidence

    # -----SOME NUMPY CAPABILITIES----

    #BROADCASTING:
    log_numerator = np.log(p) + (n_t - 1) * np.log(1 - p) #elementswise for each element for n_t (for every breach) the formula is calculated.


    ##NP WHERE with three arguments (condition,value_when_true,value_when_false).
    #n_t ==1 points here an reocurring breach right after the previous one
    log_denominator = np.where(
    n_t == 1, #this creates an boolean array [true,false] about whether the current n value is 1
    np.log(1 / n_t), #value if n_t==1
    np.log(1 / n_t) + (n_t - 1) * np.log(1 - 1 / n_t)  #value if not
    )

    # with np.sum we are recieving the sum of those elementwise calculated values
    LR_TBFI = -2 * np.sum(log_numerator - log_denominator)

    # our test defined here as a combined test of kubiec + haas
    LR_POF =kupiec_pof_single_window(start_date, end_date, realized_loss, var_value_df, var_confidence, test_confidence,detailed_output= True)["test_statistic"]

    test_statistic= LR_TBF = LR_TBFI + LR_POF

    alpha = 1 - test_confidence
    degrees_of_freedom = X_days_exceeded_number + 1

    threshold = stats.chi2.ppf(1 - alpha, df=degrees_of_freedom)


    return not (test_statistic > threshold)



## 4. Christoffersen's Independence Test (IF - Interval Forecast)

* **Core idea:** The POF test only looks at the *total* exception count, it doesn't look at *when* exceptions occur.
    * But in the real world, VaR breaches tend to **cluster** (due to volatility clustering), meaning if a breach happens today, there's a higher chance one happens tomorrow too.
    * Even if a model passes the POF test on total exception count, if breaches keep occurring back-to-back, that's a serious warning sign, because the independence assumption breaks down.

* Christoffersen's test measures exactly this: **does today's breach probability depend on yesterday's breach?**

### How it works:

You classify consecutive day-pairs into 4 categories:

| | Today: no breach | Today: breach |
|---|---|---|
| **Yesterday: no breach** | $n_{00}$ | $n_{01}$ |
| **Yesterday: breach** | $n_{10}$ | $n_{11}$ |

Then you estimate two conditional probabilities:

$$
\pi_0 = \frac{n_{01}}{n_{00}+n_{01}} \quad \text{(probability of a breach today, given no breach yesterday)}
$$

$$
\pi_1 = \frac{n_{11}}{n_{10}+n_{11}} \quad \text{(probability of a breach today, given a breach yesterday)}
$$

$$
\pi = \frac{n_{01}+n_{11}}{n_{00}+n_{01}+n_{10}+n_{11}} \quad \text{(overall/unconditional breach probability)}
$$


### null hypothesis:

* **Null hypothesis:** $\pi_0 = \pi_1 = \pi$, meaning yesterday's state has no effect on today at all (independence).

* This is again a likelihood ratio test, following the same logic as POF:
    * the denominator (unrestricted) lets $\pi_0$ and $\pi_1$ vary freely (best fit to the data), while the numerator (restricted/null) uses a single common $\pi$. If $\pi_0 \approx \pi_1$.
    * If the difference is small, LR is small. If $\pi_1 \gg \pi_0$ (breach-after-breach probability is much higher), the difference grows, LR grows, and independence is rejected.

**Why this is different from and complements POF:**

- POF: "Is the total breach count correct?" (frequency/coverage issue)
- Christoffersen IF: "Are breaches independent of each other, or do they cluster?" (dependency/clustering issue)

A model should pass both. That's why you combine them:

$$
LR_{CC} = LR_{POF} + LR_{CCI} \sim \chi^2_{(2)}
$$

This **conditional coverage (CC)** test checks both "is the breach count correct" and "are these breaches distributed independently" in one shot. Having 2 degrees of freedom makes sense because you're testing two separate hypotheses (frequency + independence) simultaneously.


In [ ]:
def christoffersen_independence_single_window(
    start_date: str | pd.Timestamp,
    end_date: str | pd.Timestamp,
    realized_loss: pd.DataFrame,
    var_value_df: pd.DataFrame,
    var_confidence: float,
    test_confidence: float = 0.95,
    detailed_output: bool = False
) -> bool | dict:

    preprocess = loss_exceeding_days(realized_loss, var_value_df, start_date, end_date)
    breach = preprocess["days_exceeded_boolean"].values.astype(int)

    today = breach[1:]
    yesterday = breach[:-1]

    n00 = np.sum((yesterday == 0) & (today == 0))
    n01 = np.sum((yesterday == 0) & (today == 1))
    n10 = np.sum((yesterday == 1) & (today == 0))
    n11 = np.sum((yesterday == 1) & (today == 1))

    pi0 = n01 / (n00 + n01)
    pi1 = n11 / (n10 + n11)
    pi = (n01 + n11) / (n00 + n01 + n10 + n11)

    def safe_log(x):
        return np.log(x) if x > 0 else 0.0

    log_numerator = (
        (n00 + n10) * safe_log(1 - pi) + (n01 + n11) * safe_log(pi)
    )

    log_denominator = (
        n00 * safe_log(1 - pi0) + n01 * safe_log(pi0) +
        n10 * safe_log(1 - pi1) + n11 * safe_log(pi1)
    )

    test_statistic = -2 * (log_numerator - log_denominator)

    alpha = 1 - test_confidence
    threshold = stats.chi2.ppf(1 - alpha, df=1)

    reject_h0 = test_statistic > threshold

    if detailed_output:
        return {
            "test_statistic": test_statistic,
            "threshold": threshold,
            "n00": n00, "n01": n01, "n10": n10, "n11": n11,
            "pi0": pi0, "pi1": pi1, "pi": pi,
            "reject_h0": reject_h0
        }

    return not reject_h0


def christoffersen_conditional_coverage_single_window(
    start_date: str | pd.Timestamp,
    end_date: str | pd.Timestamp,
    realized_loss: pd.DataFrame,
    var_value_df: pd.DataFrame,
    var_confidence: float,
    test_confidence: float = 0.95,
    detailed_output: bool = False
) -> bool | dict:

    LR_POF = kupiec_pof_single_window(
        start_date, end_date, realized_loss, var_value_df,
        var_confidence, test_confidence, detailed_output=True
    )["test_statistic"]

    LR_IND = christoffersen_independence_single_window(
        start_date, end_date, realized_loss, var_value_df,
        var_confidence, test_confidence, detailed_output=True
    )["test_statistic"]

    test_statistic = LR_POF + LR_IND

    alpha = 1 - test_confidence
    threshold = stats.chi2.ppf(1 - alpha, df=2)

    reject_h0 = test_statistic > threshold

    if detailed_output:
        return {
            "test_statistic": test_statistic,
            "threshold": threshold,
            "LR_POF": LR_POF,
            "LR_independence": LR_IND,
            "reject_h0": reject_h0
        }

    return not reject_h0